# Pretraining-strategy comparison

Which pretraining objective (neighbor-matching / contrastive / feature-prediction,
all on covid data) transfers best across the retweet-net benchmark?

Reads the shared per-task CSVs (keyed by `model` = strategy), produced by
`scripts/harness/benchmark_tasks/parse_benchmark_eval_logs.py`. Headline tasks:
**node regression** (Spearman) and **static link prediction** (ROC-AUC).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PLOT = Path('..')
STRATEGIES = ['task_transfer_covid_nm', 'task_transfer_covid_cl', 'task_transfer_covid_fp']
SHORT = {s: s.replace('task_transfer_covid_', '') for s in STRATEGIES}

reg = pd.read_csv(PLOT / 'node_regression/data/node_regression.csv')
slp = pd.read_csv(PLOT / 'static_link_prediction/data/static_link_prediction.csv')
reg = reg[(reg.split == 'test') & (reg.model.isin(STRATEGIES))].copy()
slp = slp[(slp.split == 'test') & (slp.model.isin(STRATEGIES))].copy()
for d in (reg, slp):
    d['strategy'] = d.model.map(SHORT)
print('reg', reg.shape, '| slp', slp.shape)

## Node regression — Spearman by strategy × target (mean over datasets)

In [ ]:
piv = reg.pivot_table(index='strategy', columns='target', values='spearman', aggfunc='mean').round(3)
display(piv)
ax = piv.T.plot(kind='bar', figsize=(11, 4))
ax.set_ylabel('Spearman ρ (mean over datasets)'); ax.axhline(0, color='k', lw=0.6)
ax.set_title('Regression: pretraining strategy by target'); ax.legend(title='strategy')
plt.tight_layout(); plt.show()

## Static link prediction — ROC-AUC by strategy × dataset

In [ ]:
piv = slp.pivot_table(index='strategy', columns='dataset', values='roc_auc', aggfunc='max').round(3)
display(piv)
ax = piv.T.plot(kind='bar', figsize=(11, 4))
ax.set_ylabel('ROC-AUC (test, hard negatives)'); ax.axhline(0.5, color='k', ls='--', lw=0.6)
ax.set_title('Static LP: pretraining strategy by dataset'); ax.legend(title='strategy')
plt.tight_layout(); plt.show()

## Which strategy wins? (mean primary metric per task)

In [ ]:
summary = pd.DataFrame({
    'regression_spearman_mean': reg.groupby('strategy').spearman.mean(),
    'static_lp_auc_mean': slp.groupby('strategy').roc_auc.mean(),
}).round(3)
display(summary)